In [1]:
# ======================================
# STEP 1: Load ENV
# ======================================
import os
from dotenv import load_dotenv

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")


# ======================================
# STEP 2: NVIDIA LLM
# ======================================
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(

    api_key=NVIDIA_API_KEY,

    base_url="https://integrate.api.nvidia.com/v1",

    model="meta/llama3-70b-instruct",

    temperature=0
)


# ======================================
# STEP 3: Dummy Retail Database
# ======================================
customers = {

    "C101": {

        "name": "Amit",

        "purchase_history": [

            "Laptop",

            "Mouse",

            "Keyboard"

        ]

    },

    "C102": {

        "name": "Neha",

        "purchase_history": [

            "Shoes",

            "T-shirt"

        ]

    }

}


inventory = {

    "laptop": 15,

    "mouse": 40,

    "keyboard": 25,

    "shoes": 30

}


# ======================================
# STEP 4: MCP Tools
# ======================================
def view_purchase_history(customer_id):

    if customer_id in customers:

        return f"🛒 Purchases: {customers[customer_id]['purchase_history']}"

    return "❌ customer not found"



def view_other_purchases(customer_id):

    if customer_id in customers:

        return f"📦 Customer {customer_id} purchases: {customers[customer_id]['purchase_history']}"

    return "❌ customer not found"



def approve_supplier_order():

    return "📑 Supplier order approved"



def check_inventory(product):

    product = product.lower()

    if product in inventory:

        return f"📊 {product} stock: {inventory[product]}"

    return "❌ product not found"



def manage_schedule():

    return "🗓 employee schedule updated"


# ======================================
# STEP 5: RBAC Security
# ======================================
def secure_access(

    role,

    user_id,

    requested_id,

    action

):

    if role == "manager":

        return True


    if role == "staff":

        return action in [

            "view_purchase_history",

            "view_other_purchases",

            "check_inventory"

        ]


    if role == "customer":

        return (

            user_id == requested_id

            and action == "view_purchase_history"

        )

    return False


# ======================================
# STEP 6: Intent detection via LLM
# ======================================
def decide_action(query):

    prompt = f"""

    classify into ONE:

    view_purchase_history
    view_other_purchases
    approve_supplier_order
    check_inventory
    manage_schedule

    query: {query}

    return only action name
    """

    response = llm.invoke(

        [HumanMessage(content=prompt)]

    )

    return response.content.strip().lower()


# ======================================
# STEP 7: MCP Agent
# ======================================
def retail_agent(

    message,

    role,

    user_id,

    requested_id,

    history

):

    if history is None:

        history = []


    if not user_id:

        reply = "enter customer id"

        history.append({

            "role": "assistant",

            "content": reply
        })

        return "", history


    if not requested_id:

        requested_id = user_id


    action = decide_action(message)


    if not secure_access(

        role,

        user_id,

        requested_id,

        action
    ):

        reply = "🚫 access denied"


    else:

        if action == "view_purchase_history":

            reply = view_purchase_history(requested_id)


        elif action == "view_other_purchases":

            reply = view_other_purchases(requested_id)


        elif action == "approve_supplier_order":

            reply = approve_supplier_order()


        elif action == "check_inventory":

            words = message.split()

            product = words[-1]

            reply = check_inventory(product)


        elif action == "manage_schedule":

            reply = manage_schedule()


        else:

            reply = "ask retail related question"


    history.append({

        "role": "user",

        "content": message
    })


    history.append({

        "role": "assistant",

        "content": reply
    })


    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr


with gr.Blocks() as demo:

    gr.Markdown("## 🏬 Retail Smart Assistant (RBAC + MCP)")


    role = gr.Dropdown(

        ["customer", "staff", "manager"],

        value="customer",

        label="Role"
    )


    user_id = gr.Textbox(

        value="C101",

        label="Your Customer ID"
    )


    requested_id = gr.Textbox(

        label="Target Customer ID (optional)"
    )


    chatbot = gr.Chatbot(height=400)


    msg = gr.Textbox(

        label="Ask question"
    )


    state = gr.State([])


    msg.submit(

        retail_agent,

        inputs=[

            msg,

            role,

            user_id,

            requested_id,

            state

        ],

        outputs=[msg, chatbot]

    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
